# CP2 LastFM: audit dan persiapan data

Notebook ini untuk memproses dan mengaudit data LastFM CP2 secara terpisah. Notebook **tidak menjalankan training** dan tidak menimpa file CP1.

Timestamp HetRec berasal dari event pemberian tag artis, bukan waktu mendengarkan/scrobble. Hasil CP2 harus ditafsirkan sebagai eksperimen fitur waktu aktivitas tagging.

Jalankan sel berurutan. Atur lokasi file mentah pada sel konfigurasi. Bila ingin membandingkan dengan sequence CP1, setel juga path file LastFM CP1. Output disimpan ke folder `data_cp2` di samping notebook.


In [ ]:
from pathlib import Path
from collections import Counter, OrderedDict
import csv, hashlib, json, statistics

# Sesuaikan dengan lokasi user_taggedartists-timestamps.dat.
RAW_FILE = Path(r"../raw/LastFM/user_taggedartists-timestamps.dat")
# Opsional: file sequence CP1 yang benar-benar dipakai. Contoh: Path(r"../../BSARec/src/data/LastFM.txt")
CP1_ITEMS_FILE = None
OUTPUT_DIR = Path("data_cp2")
USER_CORE, ITEM_CORE = 5, 5

print("Notebook dir:", Path.cwd())
print("Raw file:", RAW_FILE.resolve())
assert RAW_FILE.exists(), f"File sumber tidak ditemukan: {RAW_FILE.resolve()}"


## 1. Baca sumber dan pastikan skema

Data dibaca sebagai record utuh (user, artis, timestamp). Timestamp yang tampak dalam milidetik dinormalkan ke detik. Keputusan unit dicatat dalam manifest.


In [ ]:
with RAW_FILE.open("r", encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f, delimiter="\t")
    headers = {h.strip(): h for h in (reader.fieldnames or [])}
    assert {"userID", "artistID", "timestamp"}.issubset(headers), reader.fieldnames
    raw = []
    for row in reader:
        raw.append((row[headers["userID"]].strip(), row[headers["artistID"]].strip(), int(row[headers["timestamp"]])))
assert raw, "File sumber kosong"
max_ts = max(t for _, _, t in raw)
if max_ts >= 100_000_000_000:
    divisor, source_unit = 1000, "milliseconds"
else:
    divisor, source_unit = 1, "seconds"
print("Events:", len(raw), "users:", len({u for u,_,_ in raw}), "artists:", len({i for _,i,_ in raw}))
print("Timestamp unit:", source_unit, "max raw timestamp:", max_ts)


## 2. Satu interaksi per pasangan user-artis

Ikuti aturan di notebook CP1: ambil kemunculan pertama pasangan (user, artis) dalam urutan baris sumber, lalu urutkan event yang dipertahankan secara stabil berdasarkan timestamp. Jika sumber tidak tersusun kronologis, aturan ini dapat berbeda dari memilih timestamp paling awal; karena itu notebook akan melaporkan hasil perbandingan sequence lengkap dengan CP1.


In [ ]:
by_user = OrderedDict()
for row_index, (u, item, ts) in enumerate(raw):
    by_user.setdefault(u, []).append((item, ts, row_index))

# CP1 reference rule: keep the first source-row occurrence per user-item,
# then stable-sort those retained events by timestamp.
sequences = OrderedDict()
for u, records in by_user.items():
    seen = set()
    retained = []
    for item, ts, row_index in records:
        if item not in seen:
            seen.add(item)
            retained.append((item, ts // divisor, row_index))
    retained.sort(key=lambda r: r[1])
    sequences[u] = [(item, ts) for item, ts, _ in retained]

assert all(len(seq) == len({item for item,_ in seq}) for seq in sequences.values())
print("After CP1-rule user-item dedup:", sum(map(len, sequences.values())))


## 3. Filter 5-core dengan timestamp tetap melekat

Setiap putaran menghapus user dan artis yang tidak memenuhi ambang. Timestamp tetap berada pada record artis yang sama; gap baru dihitung setelah filter konvergen.


In [ ]:
while True:
    valid_users = {u for u, seq in sequences.items() if len(seq) >= USER_CORE}
    item_counts = Counter(item for u in valid_users for item, _ in sequences[u])
    valid_items = {item for item, n in item_counts.items() if n >= ITEM_CORE}
    updated = OrderedDict()
    for u, seq in sequences.items():
        if u not in valid_users:
            continue
        kept = [(item, ts) for item, ts in seq if item in valid_items]
        if len(kept) >= USER_CORE:
            updated[u] = kept
    if updated == sequences:
        break
    sequences = updated

assert sequences
user_counts = [len(s) for s in sequences.values()]
item_counts = Counter(item for seq in sequences.values() for item,_ in seq)
assert min(user_counts) >= USER_CORE and min(item_counts.values()) >= ITEM_CORE
assert all(len(s) == len({i for i,_ in s}) for s in sequences.values())
print("Users:",len(sequences),"artists:",len(item_counts),"events:",sum(user_counts))
print("Sequence avg/min/max:",round(statistics.mean(user_counts),2),min(user_counts),max(user_counts))


## 4. Buat ID, timestamp, dan gap

ID user/artis dibuat 1-based berdasarkan urutan traversal hasil filter. File `*_gaps_seconds.txt` berisi selisih detik antar-event, dengan gap pertama 0. File timestamp absolut juga disimpan untuk audit.


In [ ]:
user2id = {u:i+1 for i,u in enumerate(sequences)}
artist2id = OrderedDict()
item_seqs, time_seqs, gap_seqs = OrderedDict(), OrderedDict(), OrderedDict()
for u, seq in sequences.items():
    uid = user2id[u]
    items, times = [], []
    for artist, ts in seq:
        if artist not in artist2id:
            artist2id[artist] = len(artist2id)+1
        items.append(artist2id[artist]); times.append(ts)
    gaps = [0] + [times[i]-times[i-1] for i in range(1,len(times))]
    assert len(items)==len(times)==len(gaps)
    assert all(g >= 0 for g in gaps)
    item_seqs[uid], time_seqs[uid], gap_seqs[uid] = items, times, gaps
all_gaps = [g for seq in gap_seqs.values() for g in seq[1:]]
print("Zero gaps:",sum(g==0 for g in all_gaps),"positive gaps:",sum(g>0 for g in all_gaps))
print("Largest gap (days):",max(all_gaps,default=0)/86400)


## 5. Simpan output CP2 terpisah


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
def write_sequences(path, mapping):
    with path.open("w",encoding="utf-8",newline="\n") as f:
        for uid, vals in mapping.items():
            f.write(str(uid)+" "+" ".join(map(str,vals))+"\n")
items_file = OUTPUT_DIR/"LastFM_cp2_items.txt"
times_file = OUTPUT_DIR/"LastFM_cp2_timestamps_seconds.txt"
gaps_file = OUTPUT_DIR/"LastFM_cp2_gaps_seconds.txt"
write_sequences(items_file,item_seqs); write_sequences(times_file,time_seqs); write_sequences(gaps_file,gap_seqs)
user_map_file = OUTPUT_DIR/"LastFM_cp2_user2id.json"
artist_map_file = OUTPUT_DIR/"LastFM_cp2_artist2id.json"
user_map_file.write_text(json.dumps(user2id,ensure_ascii=False,indent=2),encoding="utf-8")
artist_map_file.write_text(json.dumps(artist2id,ensure_ascii=False,indent=2),encoding="utf-8")
def sha256(p): return hashlib.sha256(p.read_bytes()).hexdigest()
manifest = {
 "dataset":"GroupLens HetRec 2011 Last.fm 2K",
 "timestamp_semantics":"tagging event; not listening/scrobble time",
 "source":str(RAW_FILE.resolve()),"source_unit":source_unit,
 "dedup":"first source-row occurrence per user-artist pair, then stable timestamp sort",
 "k_core":{"user":USER_CORE,"artist":ITEM_CORE},
 "counts":{"users":len(user2id),"artists":len(artist2id),"interactions":sum(map(len,item_seqs.values()))},
 "gap_unit":"seconds",
 "sha256":{p.name:sha256(p) for p in [items_file,times_file,gaps_file,user_map_file,artist_map_file]}
}
manifest_file=OUTPUT_DIR/"LastFM_cp2_manifest.json"
manifest_file.write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding="utf-8")
print(json.dumps(manifest,ensure_ascii=False,indent=2))



## 6. Validasi file dan perbandingan opsional dengan CP1

Pemeriksaan ini mengecek ID baris, jumlah elemen, gap pertama, aritmetika gap, gap negatif, dan artis berulang. Bila file CP1 diberikan, bandingkan sequence lengkap; jika berbeda, jangan mengklaim preprocessing identik hanya karena jumlah totalnya sama.


In [ ]:
def read_sequences(path):
    out=OrderedDict()
    for line in path.read_text(encoding="utf-8").splitlines():
        vals=list(map(int,line.split())); out[vals[0]]=vals[1:]
    return out
I,T,G=map(read_sequences,[items_file,times_file,gaps_file])
assert list(I)==list(T)==list(G)
for uid in I:
    assert len(I[uid])==len(T[uid])==len(G[uid])
    assert G[uid][0]==0
    assert G[uid][1:]==[T[uid][i]-T[uid][i-1] for i in range(1,len(T[uid]))]
    assert min(G[uid])>=0
    assert len(I[uid])==len(set(I[uid]))
print("Internal alignment and gap arithmetic: PASS")
if CP1_ITEMS_FILE is not None:
    cp1=read_sequences(Path(CP1_ITEMS_FILE))
    print("CP1 users/events:",len(cp1),sum(map(len,cp1.values())))
    print("CP2 users/events:",len(I),sum(map(len,I.values())))
    print("Exact ordered item sequence match:",list(cp1.items())==list(I.items()))
else:
    print("CP1 comparison skipped. Set CP1_ITEMS_FILE for full sequence comparison.")


## Lanjut ke model CP2 setelah audit

Jangan training sebelum sumber, sequence/mapping, dan alignment disetujui. Rancangan minimal berikutnya adalah mengubah gap histori menjadi bucket (termasuk kategori gap nol), mempelajari embedding bucket, lalu menambahkannya ke embedding item pada posisi histori. Untuk target di posisi t, model hanya boleh menerima waktu sampai t-1.

Pertahankan objective, split, seed, training budget, dan evaluator BSARec CP1. File waktu di dataset ini mencerminkan aktivitas tagging, bukan perilaku mendengarkan. Notebook ini belum memuat model/loader karena source BSARec yang dipakai untuk CP1 tidak ada di workspace; integrasi harus mengikuti kode tersebut agar baseline tidak berubah.
